# R2AI2026 — sinh `pandas_query` trên Kaggle (T4 GPU)

Điều kiện trước khi chạy:
1. Bật **GPU T4** (Settings → Accelerator) và **Internet** (để `git clone` + tải model).
2. Upload `retrieval_results.jsonl` (build ở local bằng `python -m r2ai.retrieval.run_retrieval`) làm **Kaggle Dataset**, rồi Add Data vào notebook. File này đã nhúng sẵn CSV của các bảng candidate nên **không cần mount corpus 362MB**.
3. Sửa `REPO_URL` và `RETRIEVAL_PATH` bên dưới cho khớp.

Output: `predictions.jsonl` ghi **append + flush sau mỗi câu** trong `/kaggle/working` — tải về rồi chạy `python -m r2ai.packaging.assemble_submission` ở local (re-execute lại toàn bộ query trước khi đóng gói).

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/r2ai-stage2.git"

# Đường dẫn dataset Kaggle đã Add Data vào notebook. Kaggle mount dataset tại
# `/kaggle/input/<dataset-slug>/`, KHÔNG phải theo URL `kaggle.com/datasets/<user>/<slug>` —
# nên nếu đặt cứng sai, mọi cell sau đều fail ở FileNotFoundError. Cell 3 sẽ tự dò lại bằng
# glob nếu đường dẫn này không tồn tại, nên chỉ cần sửa khi muốn ép dùng đúng 1 file.
RETRIEVAL_PATH = "/kaggle/input/r2ai2026/retrieval_results.jsonl"
PREDICTIONS_PATH = "/kaggle/working/predictions.jsonl"
PILOT_PREDICTIONS_PATH = "/kaggle/working/predictions_pilot.jsonl"  # file riêng, không đụng resume của full run
WORK_DIR = "/kaggle/working/exec"
PILOT_N = 20  # chạy thử trước khi chạy full 1.012 câu

In [ ]:
!git clone --depth 1 $REPO_URL /kaggle/working/r2ai-stage2
%cd /kaggle/working/r2ai-stage2
!pip install -q -r requirements.txt

In [ ]:
import logging
import os
import sys

sys.path.insert(0, "/kaggle/working/r2ai-stage2")
# Phải set TRƯỚC `import torch` để có tác dụng (CUDA context init lúc import).
# Đặt CẢ HAI tên: torch mới đã đổi sang `PYTORCH_ALLOC_CONF` (chính thông báo OOM thật cũng gợi ý
# tên mới này), tên cũ `PYTORCH_CUDA_ALLOC_CONF` giữ lại cho bản torch cũ hơn.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Tắt log INFO ồn (mỗi request HTTP khi tải model từ HuggingFace Hub, numexpr) — không phải lỗi,
# chỉ là noise. WARNING/ERROR vẫn hiện đầy đủ.
for name in ("httpx", "httpcore", "numexpr", "urllib3"):
    logging.getLogger(name).setLevel(logging.WARNING)

import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
      "| số GPU:", torch.cuda.device_count())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"GPU 0: trống {free/2**30:.2f} GiB / tổng {total/2**30:.2f} GiB")
assert torch.cuda.is_available(), (
    "Không phát hiện GPU! Vào Settings (panel bên phải notebook) -> Accelerator -> chọn GPU T4 x2 "
    "(hoặc T4 x1) -> Save -> notebook sẽ restart session. Chạy lại từ đầu sau đó. Không có GPU thì "
    "load model 7B sẽ cực chậm/OOM trên CPU."
)

# Fail sớm ngay tại đây nếu cell cài đặt ở trên bị lỗi, thay vì để lỗi rơi xuống tận lúc load model 7B.
import bitsandbytes
import numpy
import pandas
import transformers

print("pandas", pandas.__version__, "| numpy", numpy.__version__,
      "| transformers", transformers.__version__, "| bitsandbytes", bitsandbytes.__version__)

# Tự dò file retrieval nếu đường dẫn đặt ở cell 1 không tồn tại (tên dataset khi Add Data
# thường khác với slug trên URL) — fail sớm ở đây kèm danh sách file thật, thay vì để lỗi
# FileNotFoundError rơi xuống giữa pilot.
import glob
import os.path

if not os.path.exists(RETRIEVAL_PATH):
    found = sorted(glob.glob("/kaggle/input/**/retrieval_results*.jsonl", recursive=True))
    print(f"Không thấy {RETRIEVAL_PATH}; tìm được: {found}")
    assert found, (
        "Chưa Add Data dataset chứa retrieval_results.jsonl vào notebook (panel phải -> Add Input). "
        f"Nội dung /kaggle/input hiện tại: {sorted(glob.glob('/kaggle/input/*'))}"
    )
    RETRIEVAL_PATH = found[0]

with open(RETRIEVAL_PATH, encoding="utf-8") as fh:
    n_questions = sum(1 for line in fh if line.strip())
print(f"retrieval: {RETRIEVAL_PATH} ({n_questions} câu)")
assert n_questions == 1012, f"Mong đợi 1.012 câu, file có {n_questions} — file retrieval có thể cũ/thiếu."

In [ ]:
# Đăng nhập HuggingFace Hub (không bắt buộc với Qwen, chỉ để bớt cảnh báo rate-limit khi tải model).
# Điền token thật vào đây SAU KHI đã import notebook này lên Kaggle — đừng commit bản đã điền token
# ngược lại về git. Bọc try/except: chạy "Save and Run All" không có người canh, token còn để
# placeholder hoặc sai thì KHÔNG được làm dừng cả notebook (Qwen tải được không cần token).
from huggingface_hub import login

try:
    login("HF_TOKEN")  # TODO: dán token HuggingFace thật vào đây trên Kaggle
    print("Đã đăng nhập HuggingFace Hub.")
except Exception as exc:
    print(f"Bỏ qua đăng nhập HF ({exc}) — vẫn chạy tiếp không token, chỉ bị giới hạn rate-limit.")

## 1. Smoke test: prompt + sandbox (không cần GPU)

`--dry-run` không nạp LLM: chỉ dựng prompt, ghi CSV ra đĩa và chạy sandbox. Bắt sớm lỗi encode CSV / đường dẫn trước khi tốn thời gian GPU.

In [ ]:
from r2ai.generation.run_generation import run

stats = run(
    retrieval_path=RETRIEVAL_PATH,
    out_path="/kaggle/working/predictions_dryrun.jsonl",
    work_dir=WORK_DIR,
    limit=3,
    dry_run=True,
    resume=False,
)
assert stats["exec_ok"] > 0, (
    f"Smoke test thất bại (exec_ok={stats['exec_ok']}/{stats['attempted']}) — xem traceback/lỗi phía "
    "trên. 'Save and Run All' dừng ở đây, không tốn thời gian chạy tiếp pilot/full khi plumbing hỏng."
)
print("smoke test OK:", stats)

## 2. Load model (1 lần) + pilot 20 câu — đo wall-clock/câu

Model được load 1 lần ở đây và tái sử dụng cho cả pilot và bước full ở dưới (không load lại 2 lần). Ước lượng tổng thời gian cho 1.012 câu so với giới hạn session Kaggle (~9-12h). Nếu quá lâu: đổi sang `Qwen/Qwen2.5-Coder-3B-Instruct` (sửa `configs/baseline.yaml`, chạy lại từ cell load model) hoặc giảm `candidates_in_prompt`.

Chạy "Save and Run All": pilot thất bại hoặc `exec_ok=0` sẽ **dừng notebook tại đây** — mẫu lỗi thật (`exec_error`) được in ngay dưới để chẩn đoán, không cần chạy thêm cell nào khác.

In [ ]:
import json
import time

import torch

from r2ai.config import load_config
from r2ai.generation.run_generation import LocalLLM, run

gen_cfg = load_config()["generation"]
llm = LocalLLM(
    gen_cfg["model"],
    load_in_4bit=bool(gen_cfg["load_in_4bit"]),
    max_new_tokens=int(gen_cfg["max_new_tokens"]),
    temperature=float(gen_cfg["temperature"]),
)

start = time.time()
stats = run(
    llm=llm,
    retrieval_path=RETRIEVAL_PATH,
    out_path=PILOT_PREDICTIONS_PATH,
    work_dir=WORK_DIR,
    limit=PILOT_N,
    resume=False,  # file riêng cho pilot -> luôn chạy lại từ đầu, không dây với resume của full run
)
elapsed = time.time() - start

print(f"pilot: {stats}")
print(f"  OOM khi generate: {stats['generate_oom']} lần | phải thử lại prompt ngắn hơn: {stats['generate_retried']} lần")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"  GPU 0 sau pilot: trống {free/2**30:.2f} GiB / {total/2**30:.2f} GiB | attention = {llm.attn_implementation}")
print(f"{elapsed / PILOT_N:.1f}s/câu -> ước tính {elapsed / PILOT_N * 1012 / 3600:.1f}h cho 1.012 câu")

rows = [json.loads(line) for line in open(PILOT_PREDICTIONS_PATH, encoding="utf-8") if line.strip()]
failed = [r for r in rows if not r["exec_ok"]]
for row in failed[:5]:
    print(f"--- id={row['id']} exec_error ---")
    print((row["exec_error"] or "")[:300])
    print("pandas_query (300 ký tự cuối):", row["pandas_query"][-300:])

assert stats["exec_ok"] > 0, (
    f"Pilot THẤT BẠI (exec_ok={stats['exec_ok']}/{stats['attempted']}) — xem exec_error in ở trên. "
    "'Save and Run All' dừng tại đây, không chạy tiếp bước full 1.012 câu (~nhiều giờ)."
)

## 3. Chạy full (resume được)

`run()` mặc định `resume=True`: bỏ qua các id đã có trong `predictions.jsonl`, nên chạy lại cell này sau khi session bị ngắt là tiếp tục từ chỗ dừng (dùng lại `llm` đã load ở cell trên, không load lại).

In [ ]:
from r2ai.generation.run_generation import run

stats = run(llm=llm, retrieval_path=RETRIEVAL_PATH, out_path=PREDICTIONS_PATH, work_dir=WORK_DIR)
print(f"full run: {stats}")

In [ ]:
import json
from collections import Counter

rows = [json.loads(line) for line in open(PREDICTIONS_PATH, encoding="utf-8") if line.strip()]
print("tổng:", len(rows), "| id duy nhất:", len({r["id"] for r in rows}))
print("exec_ok:", Counter(r["exec_ok"] for r in rows))
for row in [r for r in rows if not r["exec_ok"]][:5]:
    print(row["id"], "->", (row["exec_error"] or "")[:200])

Tải `predictions.jsonl` về máy local, đặt vào `data/interim/`, rồi:

```bash
python -m r2ai.packaging.assemble_submission   # join + re-execute ở local
python -m r2ai.packaging.zip_submission        # validate + zip
```